# CSE 251B Blended-Data Pipeline

This notebook prepares blended train shards, trains the nano model with root `val.bin` validation, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [1]:
!nvidia-smi

Wed May 27 21:13:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'blend'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'blend'

RUN_NAME = 'blend_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

Mounted at /content/drive
Drive dataset: /content/drive/MyDrive/251B/dataset/blend
Drive results: /content/drive/MyDrive/251B/results
Repo: /content/milestone
Local dataset: /content/milestone/data/blend
Local run results: /content/milestone/results/blend_nano_20260527_211352
Drive run results: /content/drive/MyDrive/251B/results/blend_nano_20260527_211352


## 2. Prepare Repository

In [3]:
!rm -rf /content/milestone
!git clone -b v3 https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Cloning into '/content/milestone'...
remote: Enumerating objects: 833, done.
remote: Counting objects: 100% (833/833), done.
remote: Compressing objects: 100% (392/392), done.
remote: Total 833 (delta 464), reused 792 (delta 434), pack-reused 0 (from 0)
Receiving objects: 100% (833/833), 7.82 MiB | 541.00 KiB/s, done.
Resolving deltas: 100% (464/464), done.
/content/milestone
## v3...origin/v3


## 3. Install Dependencies

In [4]:
!pip install -q datasets tiktoken tqdm

In [5]:
!pip install -q "datasets==2.19.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 20.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## 4. Copy Or Prepare Training Blend

In [6]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

BLEND_SOURCES = {
    'fineweb': {'dir': 'fineweb_edu', 'prefix': 'fineweb', 'min_shards': 20},
    'wikipedia': {'dir': 'wikipedia', 'prefix': 'wikipedia', 'min_shards': 9},
    'arxiv': {'dir': 'papers_arxiv', 'prefix': 'papers_arxiv', 'min_shards': 5},
    'pubmed': {'dir': 'papers_pubmed', 'prefix': 'papers_pubmed', 'min_shards': 2},
    'books': {'dir': 'books', 'prefix': 'books', 'min_shards': 7},
}

def count_shards(path):
    path = Path(path)
    return {
        name: len(sorted((path / spec['dir']).glob(f"{spec['prefix']}_train_*.bin")))
        for name, spec in BLEND_SOURCES.items()
    }

def format_counts(counts):
    return ', '.join(f'{name}={counts.get(name, 0)}' for name in BLEND_SOURCES)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob('*_train_*.bin'))

def has_ready_dataset(path):
    counts = count_shards(path)
    return all(counts[name] >= spec['min_shards'] for name, spec in BLEND_SOURCES.items())

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_counts = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: {format_counts(drive_counts)}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR):
    log('Copying blended shards from Drive to repo data/blend...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is incomplete. Preparing blended shards under repo data/blend...')
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    for source_name, source_spec in BLEND_SOURCES.items():
        cmd = [
            'python', '-u', 'data/blend.py',
            '--data-root', str(LOCAL_DATASET_DIR),
            '--sources', source_name,
            '--streaming',
            '--shard-size', str(SHARD_SIZE),
            '--max-shards-per-source', str(source_spec['min_shards']),
            '--num-proc', str(NUM_PROC),
        ]
        log('Running: ' + ' '.join(cmd))
        with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
            proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
            ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'Dataset preparation for {source_name} failed with exit code {ret}. See {DATASET_LOG}')
        log(f'Copying prepared {source_name} shards back to Drive...')
        sync_dir(LOCAL_DATASET_DIR / source_spec['dir'], DRIVE_DATASET_DIR / source_spec['dir'])
    log('Copying prepared blended shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_counts = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: {format_counts(local_counts)}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR), 'Blended dataset is not ready in repo data/blend.'

[2026-05-27 21:14:53] Drive shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB
[2026-05-27 21:14:53] Copying blended shards from Drive to repo data/blend...
[2026-05-27 21:21:57] Local shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB


## 5. Train

In [7]:
TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

import importlib.util
spec = importlib.util.spec_from_file_location('train_config', REPO_DIR / 'config' / 'config.py')
train_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_config)
print('Optimizer:', getattr(train_config, 'optimizer', 'adamw'))
print('Muon lr:', getattr(train_config, 'muon_lr', None))

cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
print('Run name:', RUN_NAME)
print('Command:', ' '.join(cmd))

with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')

Optimizer: muon
Muon lr: 0.02
Run name: blend_nano_20260527_211352
Command: python -u train.py --config config/config.py
device: cuda
sequence length: 1024
gradient accumulation steps: 33
train data mix schedule: fineweb_edu=23, wikipedia=4, papers=3, books=3
train data mix repeats per optimizer step: 1
validation data: /content/milestone/val.bin
parameters: 99,027,200
optimizer: muon | muon tensors: 60 (66,846,720 params) | adamw fallback tensors: 26 (32,180,480 params)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2212: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
step     0 | val loss 10.9218 | seq_len 1024
step     0 | train loss 10.943544 | adamw_lr 7.5000e-07 | muon_lr 5.0000e-05 | norm 7.9346 | seq_len 1024 | 9436 tok/s
step    10 | train loss 10.607847 | adamw_lr 8.2500e-06 | muon_lr 5.5000e-04 | norm 7.7112 | seq_len 1024 | 182349 tok/s
step    20 | train loss 9.96

## 6. Run TA Evaluation

In [8]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

Command: python -u evaluate.py --model_dir /content/milestone --checkpoint_filename checkpoints/checkpoint.pt --data /content/milestone/val.bin --device cuda --batch_size 1 --output_json /content/milestone/results/blend_nano_20260527_211352/eval_val.json

--- Evaluating local model: /content/milestone ---
Loading model from /content/milestone/checkpoints/checkpoint.pt...
Total parameters:       99,027,200
Trainable parameters:   99,027,200
Evaluating on /content/milestone/val.bin (block_size=1024)...

  Perplexity:          24.1712
  Avg loss (nats):     3.185162
  Tokens evaluated:    5,170,176
  Total parameters:    99,027,200
  Eval time:           83.6s

Results written to /content/milestone/results/blend_nano_20260527_211352/eval_val.json


## 7. Package Results And Copy To Drive

In [9]:
submission_files = {
    REPO_DIR / 'checkpoints' / 'checkpoint.pt': LOCAL_RESULTS_DIR / 'checkpoint.pt',
    REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
]

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

Local results: /content/milestone/results/blend_nano_20260527_211352
Drive results: /content/drive/MyDrive/251B/results/blend_nano_20260527_211352
Packaged files:
 - checkpoint.pt
 - checkpoint_metadata.pt
 - config.py
 - dataset_prepare.log
 - eval_val.json
 - model.py
 - train.log
 - train_stdout.log


# 8. Exit Colab

In [ ]:
from google.colab import runtime
runtime.unassign()